# Optimisers

Backprop gives the gradient. The optimiser decides what to do with it. Plain gradient descent steps straight downhill, and that is a poor plan on long narrow valleys, which is what most loss surfaces are. This notebook builds SGD, momentum, RMSprop and Adam on one 2D surface so you can see the paths, then compares them in Keras on MNIST.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
from common.dl import set_seed, setup_keras

setup_keras("torch")
import keras
import matplotlib.pyplot as plt
import numpy as np

from common.datasets import load_mnist

SEED = 42
set_seed(SEED)
keras.utils.set_random_seed(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)

## 1. Intuition

Picture a ball in a curved, narrow valley. The steepest direction points across the valley, not along it. Plain gradient descent bounces from wall to wall and creeps forward. Three fixes, each one idea:

- **Momentum** keeps a running velocity. Bounces across the valley cancel, and the steady push along it adds up.
- **RMSprop** divides each step by the recent size of that weight's gradient. Steep directions get small steps, flat ones get large steps.
- **Adam** does both, and corrects for starting at zero.

## 2. Maths and cost function

With gradient $g_t$, learning rate $\eta$ and weights $\theta$:

$$\text{SGD: }\theta\leftarrow\theta-\eta g_t\qquad \text{Momentum: }v\leftarrow\beta v+g_t,\ \theta\leftarrow\theta-\eta v$$

$$\text{RMSprop: }s\leftarrow\rho s+(1-\rho)g_t^2,\ \theta\leftarrow\theta-\eta\,\frac{g_t}{\sqrt s+\epsilon}$$

$$\text{Adam: }m\leftarrow\beta_1m+(1-\beta_1)g_t,\ s\leftarrow\beta_2s+(1-\beta_2)g_t^2,\ \theta\leftarrow\theta-\eta\,\frac{m/(1-\beta_1^t)}{\sqrt{s/(1-\beta_2^t)}+\epsilon}$$

The test surface is a scaled Rosenbrock valley, $J(x,y)=(1-x)^2+10(y-x^2)^2$, with its minimum of 0 at (1, 1). The valley is curved and about 10 times steeper across than along.

## 3. From scratch

Each optimiser is a function of the weights, the gradient and its own state. All four start at (-1.5, 1.5) and run 300 steps. SGD and momentum share a learning rate of 0.002 so the effect of the velocity is isolated.

In [ ]:
A = 10.0


def surface(p: np.ndarray) -> float:
    x, y = p
    return (1 - x) ** 2 + A * (y - x**2) ** 2


def grad(p: np.ndarray) -> np.ndarray:
    x, y = p
    return np.array([-2 * (1 - x) - 4 * A * x * (y - x**2), 2 * A * (y - x**2)])


def sgd(p, g, st, t, lr):
    return p - lr * g


def momentum(p, g, st, t, lr, beta=0.9):
    st["v"] = beta * st.get("v", 0) + g
    return p - lr * st["v"]


def rmsprop(p, g, st, t, lr, rho=0.9, eps=1e-8):
    st["s"] = rho * st.get("s", 0) + (1 - rho) * g**2
    return p - lr * g / (np.sqrt(st["s"]) + eps)


def adam(p, g, st, t, lr, b1=0.9, b2=0.999, eps=1e-8):
    st["m"] = b1 * st.get("m", 0) + (1 - b1) * g
    st["s"] = b2 * st.get("s", 0) + (1 - b2) * g**2
    m_hat, s_hat = st["m"] / (1 - b1**t), st["s"] / (1 - b2**t)  # fix the zero start
    return p - lr * m_hat / (np.sqrt(s_hat) + eps)


def run(step_fn, steps: int = 300, start=(-1.5, 1.5), **kw) -> np.ndarray:
    p, state, path = np.array(start, dtype=float), {}, [np.array(start, dtype=float)]
    for t in range(1, steps + 1):
        p = step_fn(p, grad(p), state, t, **kw)
        path.append(p)
    return np.array(path)


runs = {
    "SGD": run(sgd, lr=0.002),
    "Momentum": run(momentum, lr=0.002),
    "RMSprop": run(rmsprop, lr=0.02),
    "Adam": run(adam, lr=0.1),
}
for name, path in runs.items():
    print(f"{name:>8}: J after 300 steps = {surface(path[-1]):.4f}")

In [ ]:
gx, gy = np.meshgrid(np.linspace(-2, 2, 200), np.linspace(-0.5, 2.5, 200))
zz = (1 - gx) ** 2 + A * (gy - gx**2) ** 2
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].contour(gx, gy, zz, levels=np.logspace(-1, 2.5, 15), cmap="gray", alpha=0.5)
for name, path in runs.items():
    ax[0].plot(path[:, 0], path[:, 1], label=name, lw=1.5)
    ax[1].semilogy([surface(p) + 1e-12 for p in path], label=name)
ax[0].plot(1, 1, "k*", ms=12)
ax[0].legend()
ax[0].set_title("paths")
ax[1].set_title("J per step (log scale)")
ax[1].set_xlabel("step")
plt.show()

SGD at this rate is still on the valley wall after 300 steps (J = 3.1). Momentum, at the same rate, reaches 0.001 because the along-valley push accumulates. Adam gets to 0.0001. RMSprop is slower to settle (0.0096) since its step size does not shrink as it arrives, so it jitters near the minimum. Adam's averaged gradient damps that.

## 4. Library version

Keras ships all four. We train the same 128-unit MNIST network, from the same initial weights, with each. Each optimiser gets a typical learning rate for itself, because the rates are not comparable: SGD needs about 10 times the rate of Adam.

In [ ]:
x_tr, y_tr, x_te, y_te = load_mnist(5000, 1000, seed=SEED)
x_tr, x_te = x_tr.reshape(len(x_tr), -1), x_te.reshape(len(x_te), -1)


def make_net(optimizer) -> keras.Model:
    set_seed(SEED)
    model = keras.Sequential(
        [
            keras.Input((784,)),
            keras.layers.Dense(128, activation="relu"),
            keras.layers.Dense(10, activation="softmax"),
        ]
    )
    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


optimisers = {
    "SGD 0.01": keras.optimizers.SGD(0.01),
    "Momentum 0.01": keras.optimizers.SGD(0.01, momentum=0.9),
    "RMSprop 0.001": keras.optimizers.RMSprop(0.001),
    "Adam 0.001": keras.optimizers.Adam(0.001),
}
histories = {}
for name, opt in optimisers.items():
    model = make_net(opt)
    histories[name] = model.fit(x_tr, y_tr, epochs=8, batch_size=128, verbose=0).history
    print(
        f"{name:>14}: final train loss {histories[name]['loss'][-1]:.3f}, test acc {model.evaluate(x_te, y_te, verbose=0)[1]:.3f}"
    )
for name, h in histories.items():
    plt.plot(h["loss"], label=name)
plt.legend()
plt.xlabel("epoch")
plt.title("training loss")
plt.show()

The order matches the 2D picture: plain SGD is slowest (81% test accuracy), momentum is at 90%, and RMSprop and Adam are at 91%. Adam is the usual first choice because it needs less tuning, not because it always wins. SGD with momentum often generalises as well or better on image models, given time to tune.

## 5. Tuning

The learning rate matters more than the optimiser choice. Sweep it for Adam and SGD with momentum.

In [ ]:
for name, make_opt, rates in [
    ("Adam", lambda lr: keras.optimizers.Adam(lr), (0.0001, 0.001, 0.01, 0.1)),
    ("Momentum", lambda lr: keras.optimizers.SGD(lr, momentum=0.9), (0.001, 0.01, 0.1, 1.0)),
]:
    for lr in rates:
        model = make_net(make_opt(lr))
        model.fit(x_tr, y_tr, epochs=5, batch_size=128, verbose=0)
        loss, acc = model.evaluate(x_te, y_te, verbose=0)
        print(f"{name:>8} lr={lr:<7}: test acc {acc:.3f}")

Each optimiser has a band of about one order of magnitude where it works. Too low and it has not arrived after 5 epochs (Adam at 0.0001 reaches 82%). Too high and accuracy collapses (Adam at 0.1 reaches 48%). Adam's band sits between 0.001 and 0.01, and momentum SGD's between 0.01 and 0.1.

## 6. What breaks

Every method diverges if the rate is too large. For plain gradient descent on a quadratic, the step must be below $2/L$, where $L$ is the steepest curvature. On our valley that is a rate near 0.02 to 0.03. Try 0.04.

In [ ]:
path = run(sgd, steps=30, lr=0.04)
print("J at steps 0, 5, 10, 20, 30:", [f"{surface(path[i]):.3g}" for i in (0, 5, 10, 20, 30)])

The loss grows from 12 to 6e38 in five steps and then overflows to `nan`. The same thing in Keras shows up as a loss of `nan` after the first epoch. If you see `nan`, lower the learning rate by 10 before suspecting anything else.